# G2: repository issue flow (`02_repository_issue_flow`)

**Question.** For each repository and day: how many issues were opened, closed and reopened, and how long did the closed ones take from creation to close?

This notebook builds the Gold table `github_lakehouse.gold.repository_issue_flow`.

Contract: `docs/gold_contracts.md`, section 10. The way a Gold table is refreshed is in section 6.

| | |
|---|---|
| Reads | `silver.issue_events` only |
| Writes | `gold.repository_issue_flow`, its staging copy `gold_staging.repository_issue_flow`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per repository per UTC day on which it had at least one issue event |
| Key | `repo_id`, `activity_date` |

## The rules of this table

- **Events and issues are different things.** An issue that is opened and closed on the same day gives two events and one issue. The table has both numbers.
- **Resolution hours** use only rows with the action `closed`, both times known, and a close time that is not earlier than the creation time.
- **Resolution time starts at the original creation**, not at the last reopen. An issue that is closed twice gives two close actions and two durations.
- **A duration belongs to the day of the close event.** It can be far longer than seven days.
- **An issue is identified** by `issue_id`. If that is missing, by `issue_number` in its repository. If both are missing, the event still counts as an event, but not as an issue.
- **NULL is not zero.** A day without an eligible close event has no average and no median: both are NULL, and the eligible count next to them is 0.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Affected days | Decides which days this run rebuilds |
| 4 | Read Silver | Only the columns the table needs, only the affected days |
| 5 | Build | Hours and issue key per event, counts per repository and day, the name label |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy |
| 8 | Publish | Replaces the affected days in the Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | All days are rebuilt and the whole Gold table is replaced | `YYYY-MM-DD`: only the days touched by that source date are rebuilt and replaced |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_repository_issue_flow"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The four table names are set **only** in this cell. Later cells never change them, otherwise a test run could write to production.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_TABLE = "github_lakehouse.silver.issue_events"
    GOLD_TABLE = "github_lakehouse.gold.repository_issue_flow"
    STAGING_TABLE = "github_lakehouse.gold_staging.repository_issue_flow"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_TABLE = f"github_lakehouse.{test_schema}.issue_events"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.repository_issue_flow"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.repository_issue_flow_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

print("Silver table :", SILVER_TABLE)
print("Gold table   :", GOLD_TABLE)
print("Staging table:", STAGING_TABLE)
print("Audit table  :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`, and the Silver table by `Silver_layer/03_issue_events`. If one of them does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

if not spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        f"{SILVER_TABLE} does not exist. Run the Silver notebook 03_issue_events first."
    )

print("Setup found:", AUDIT_TABLE)
print("Source found:", SILVER_TABLE)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `activity_date` | Key. The UTC date of the events |
| `repo_name` | Label: the latest name seen that day |
| `issue_event_count` | All issue events that day |
| `issue_action_known_rows` | Events with a known action |
| `issue_opened_actions` | Events with the action `opened` |
| `issue_closed_actions` | Events with the action `closed` |
| `issue_reopened_actions` | Events with the action `reopened` |
| `issue_other_action_rows` | Events with a known action that is none of the three. Expected 0 |
| `issue_close_duration_eligible_rows` | Close events with usable times. The base of the two columns below |
| `avg_resolution_hours` | Average hours from creation to close. NULL when there is no eligible row |
| `p50_resolution_hours` | Median hours from creation to close. NULL when there is no eligible row |
| `issues_with_usable_key_rows` | Events where the issue can be identified |
| `distinct_issues_observed` | Different issues seen that day |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and label
        repo_id                             BIGINT,
        activity_date                       DATE,
        repo_name                           STRING,

        -- Events by action
        issue_event_count                   BIGINT,
        issue_action_known_rows             BIGINT,
        issue_opened_actions                BIGINT,
        issue_closed_actions                BIGINT,
        issue_reopened_actions              BIGINT,
        issue_other_action_rows             BIGINT,

        -- Time from creation to close
        issue_close_duration_eligible_rows  BIGINT,
        avg_resolution_hours                DOUBLE,
        p50_resolution_hours                DOUBLE,

        -- Issues
        issues_with_usable_key_rows         BIGINT,
        distinct_issues_observed            BIGINT,

        -- Technical columns
        _gold_run_id                        STRING,
        _gold_updated_at                    TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Affected days

A Gold row is a summary of a whole day. When one new event arrives for a day, the row of that day has to be calculated again from **all** events of that day. So the first question of every run is: which days does this run have to rebuild?

### 3.1 The Silver rows in the scope of this run

- With a `source_date`, the scope is the Silver rows that were loaded from that source date.
- With a blank date, the scope is all Silver rows.

A row without an event date cannot be placed on a day. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_row = spark.sql(f"""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM {SILVER_TABLE}
    WHERE {scope_filter}
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter                :", scope_filter)
print("Silver rows in the scope    :", rows_in_scope)
print("Rows without an event date  :", rows_without_event_date)

### 3.2 The days these rows belong to

The affected days are the event dates of the rows in the scope. In the week that is loaded, the event date and the source date of a row are always the same day (measured by the profiling run). So a run for one source date rebuilds exactly one day.

In [0]:
day_rows = spark.sql(f"""
    SELECT DISTINCT event_date
    FROM {SILVER_TABLE}
    WHERE {scope_filter}
      AND event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

print("Affected days:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)

### 3.3 The day list as SQL text

The later steps need the affected days inside a SQL `IN (...)` list, for example `DATE'2024-02-01', DATE'2024-02-02'`. The same text is used to read Silver, to check the result and to publish, so all three work on exactly the same days.

When there is no affected day, the text is `NULL`. `IN (NULL)` is never true, so the later queries simply find no row.

In [0]:
if len(affected_dates) == 0:
    affected_days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    affected_days_sql = ", ".join(day_texts)

print("Day list:", affected_days_sql)

## 4. Read Silver

All Silver rows of the affected days are read, **not only the rows of this source date**. A row that arrived earlier for the same day must be counted again, otherwise the new Gold row would be too small.

Only the columns this table needs are selected. `event_date` gets its Gold name `activity_date`.

Spark does not read anything yet. A temporary view is only a name for a query. The work is done in step 6, when the result is written.

In [0]:
issue_rows = spark.sql(f"""
    SELECT
        event_id,
        event_timestamp,
        event_date  AS activity_date,
        repo_id,
        repo_name,
        issue_action,
        issue_id,
        issue_number,
        issue_created_at,
        issue_closed_at
    FROM {SILVER_TABLE}
    WHERE event_date IN ({affected_days_sql})
""")
issue_rows.createOrReplaceTempView("issue_rows")

### 4.1 How many Silver rows these days have

The rows are counted directly on the Silver table, not on the view above. So this number does not depend on the steps that follow.

It goes into the audit row. The quality gates compare it with the sum of `issue_event_count`: every Silver row of the affected days must be counted exactly once.

In [0]:
silver_row_count = spark.sql(f"""
    SELECT COUNT(*) AS silver_rows
    FROM {SILVER_TABLE}
    WHERE event_date IN ({affected_days_sql})
""").collect()[0]["silver_rows"]

print("Silver rows on the affected days:", silver_row_count)

## 5. Build

### 5.1 Resolution hours and issue key of each event

Two values are worked out for every event. The row count does not change in this step.

**`resolution_hours`** is filled only for an eligible close event:

1. the action is `closed`,
2. the creation time and the close time are both known,
3. the close time is not earlier than the creation time.

For every other event it stays NULL. A `CASE` without `ELSE` gives NULL.

The hours are the seconds between the creation time and the close time, divided by 3,600. `unix_timestamp` turns a time into a number of seconds. The time of the event itself is not used. Example: created 1 December 2023 12:00, closed 5 February 2024 12:00 is 66 days, which is 1,584 hours.

Why the action must be `closed`: an event can carry a close time although its action is `opened` or `reopened`. Without this rule one closed issue could be measured twice.

**`issue_key`** identifies the issue: the `issue_id`, or the `issue_number` when the ID is missing. The number is unique only inside one repository. That is enough here, because the next step counts the keys per repository. If both are missing, the key stays NULL.

In [0]:
issue_events_prepared = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        issue_action,

        CASE
            WHEN issue_action = 'closed'
             AND issue_created_at IS NOT NULL
             AND issue_closed_at IS NOT NULL
             AND issue_closed_at >= issue_created_at
            THEN (unix_timestamp(issue_closed_at) - unix_timestamp(issue_created_at)) / 3600
        END AS resolution_hours,

        CASE
            WHEN issue_id IS NOT NULL
            THEN concat('id:', CAST(issue_id AS STRING))
            WHEN issue_number IS NOT NULL
            THEN concat('number:', CAST(issue_number AS STRING))
        END AS issue_key

    FROM issue_rows
""")
issue_events_prepared.createOrReplaceTempView("issue_events_prepared")

### 5.2 Counts per repository and day

One `GROUP BY` gives one row per repository and day. Every number of the table comes from the same rows, so one grouping is enough.

How the functions treat NULL:

- `COUNT(*)` counts every row.
- `count_if(condition)` counts the rows where the condition is true. A row with a NULL action is true for none of the conditions.
- `COUNT(column)` counts only the rows where the column is not NULL. So `COUNT(resolution_hours)` is the number of eligible close events.
- `AVG` and `median` use only the rows where the value is not NULL. With no such row they give NULL, not 0.
- `COUNT(DISTINCT issue_key)` counts every issue once, however many events it had that day.

`median` is the exact median. With an even number of values it is the average of the two middle values: the median of 1, 2, 3 and 10 hours is 2.5 hours.

`NOT IN ('opened', 'closed', 'reopened')` finds a known action that is none of the three. None is expected.

The average and the median are not rounded here. They are rounded once, in step 5.4.

In [0]:
issue_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,

        COUNT(*)                                    AS issue_event_count,
        count_if(issue_action IS NOT NULL)          AS issue_action_known_rows,
        count_if(issue_action = 'opened')           AS issue_opened_actions,
        count_if(issue_action = 'closed')           AS issue_closed_actions,
        count_if(issue_action = 'reopened')         AS issue_reopened_actions,
        count_if(issue_action NOT IN ('opened', 'closed', 'reopened'))
                                                    AS issue_other_action_rows,

        COUNT(resolution_hours)                     AS issue_close_duration_eligible_rows,
        AVG(resolution_hours)                       AS avg_resolution_hours_exact,
        median(resolution_hours)                    AS p50_resolution_hours_exact,

        COUNT(issue_key)                            AS issues_with_usable_key_rows,
        COUNT(DISTINCT issue_key)                   AS distinct_issues_observed

    FROM issue_events_prepared
    GROUP BY repo_id, activity_date
""")
issue_counts.createOrReplaceTempView("issue_counts")

### 5.3 The `repo_name` label

A repository can be renamed, so `repo_name` is only a label and never a key. The label of a day is the name on the **latest** event of that repository on that day.

`ROW_NUMBER` numbers the events of each repository and day, the latest first. Number 1 is the one we keep.

- Events without a name are left out first, so a missing name is skipped.
- If two events have the same time, the one with the larger `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.

In [0]:
repo_names = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        repo_name
    FROM (
        SELECT
            repo_id,
            activity_date,
            repo_name,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS name_rank
        FROM issue_rows
        WHERE repo_name IS NOT NULL
    ) ranked_names
    WHERE name_rank = 1
""")
repo_names.createOrReplaceTempView("repo_names")

### 5.4 The Gold rows

The counts and the label are put together. The columns are in the order of the Gold table.

- `LEFT JOIN` keeps a repository-day whose events all have no name. Its label is NULL.
- The average and the median are rounded to 2 decimals here, once.
- The two technical columns say which run wrote the row and when.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        c.repo_id,
        c.activity_date,
        n.repo_name,

        c.issue_event_count,
        c.issue_action_known_rows,
        c.issue_opened_actions,
        c.issue_closed_actions,
        c.issue_reopened_actions,
        c.issue_other_action_rows,

        c.issue_close_duration_eligible_rows,
        ROUND(c.avg_resolution_hours_exact, 2)  AS avg_resolution_hours,
        ROUND(c.p50_resolution_hours_exact, 2)  AS p50_resolution_hours,

        c.issues_with_usable_key_rows,
        c.distinct_issues_observed,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at

    FROM issue_counts c
    LEFT JOIN repo_names n
      ON  n.repo_id = c.repo_id
      AND n.activity_date = c.activity_date
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM gold_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 7. Quality gates

The checks run on the staging copy. If one of them fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and day appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `activity_date` |
| `COUNT_IS_NULL` | One of the count columns is missing. A count is 0 when nothing was counted, never NULL |
| `ACTIONS_DO_NOT_ADD_UP` | Opened + closed + reopened + other is not the number of known actions |
| `KNOWN_ACTIONS_ABOVE_EVENTS` | A row has more known actions than events |
| `ELIGIBLE_ROWS_ABOVE_CLOSED_ACTIONS` | A row has more duration eligible rows than close actions |
| `NEGATIVE_DURATION` | An average or a median is below 0 |
| `DURATION_DOES_NOT_FIT_ELIGIBLE_ROWS` | A row has an average or a median without eligible rows, or eligible rows without them |
| `ISSUE_COUNTS_OUT_OF_ORDER` | Distinct issues ≤ rows with a usable key ≤ all events is not true |
| `EVENT_SUM_DIFFERS_FROM_SILVER` | The sum of `issue_event_count` is not the number of Silver rows on the affected days |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, activity_date
            FROM {STAGING_TABLE}
            GROUP BY repo_id, activity_date
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last two values are the row count and the sum of all events.

`COALESCE(SUM(...), 0)`: the sum of no rows is NULL. For the comparison with Silver it has to be 0.

In [0]:
failed_checks = []
staged_row_count = 0
staged_event_sum = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR activity_date IS NULL)
                AS rows_without_key,

            count_if(issue_event_count IS NULL
                  OR issue_action_known_rows IS NULL
                  OR issue_opened_actions IS NULL
                  OR issue_closed_actions IS NULL
                  OR issue_reopened_actions IS NULL
                  OR issue_other_action_rows IS NULL
                  OR issue_close_duration_eligible_rows IS NULL
                  OR issues_with_usable_key_rows IS NULL
                  OR distinct_issues_observed IS NULL)
                AS rows_with_missing_count,

            count_if(issue_opened_actions + issue_closed_actions
                     + issue_reopened_actions + issue_other_action_rows
                     <> issue_action_known_rows)
                AS rows_where_actions_do_not_add_up,

            count_if(issue_action_known_rows > issue_event_count)
                AS rows_with_known_actions_above_events,

            count_if(issue_close_duration_eligible_rows > issue_closed_actions)
                AS rows_with_eligible_above_closed,

            count_if(avg_resolution_hours < 0 OR p50_resolution_hours < 0)
                AS rows_with_negative_duration,

            count_if(
                (issue_close_duration_eligible_rows = 0
                 AND (avg_resolution_hours IS NOT NULL OR p50_resolution_hours IS NOT NULL))
                OR
                (issue_close_duration_eligible_rows > 0
                 AND (avg_resolution_hours IS NULL OR p50_resolution_hours IS NULL))
            )   AS rows_where_duration_does_not_fit,

            count_if(distinct_issues_observed > issues_with_usable_key_rows
                  OR issues_with_usable_key_rows > issue_event_count)
                AS rows_with_issue_counts_out_of_order,

            COUNT(*)                             AS staged_rows,
            COALESCE(SUM(issue_event_count), 0)  AS staged_event_sum

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]
    staged_event_sum = gate_row["staged_event_sum"]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_actions_do_not_add_up"] > 0:
        failed_checks.append("ACTIONS_DO_NOT_ADD_UP")
    if gate_row["rows_with_known_actions_above_events"] > 0:
        failed_checks.append("KNOWN_ACTIONS_ABOVE_EVENTS")
    if gate_row["rows_with_eligible_above_closed"] > 0:
        failed_checks.append("ELIGIBLE_ROWS_ABOVE_CLOSED_ACTIONS")
    if gate_row["rows_with_negative_duration"] > 0:
        failed_checks.append("NEGATIVE_DURATION")
    if gate_row["rows_where_duration_does_not_fit"] > 0:
        failed_checks.append("DURATION_DOES_NOT_FIT_ELIGIBLE_ROWS")
    if gate_row["rows_with_issue_counts_out_of_order"] > 0:
        failed_checks.append("ISSUE_COUNTS_OUT_OF_ORDER")
    if staged_event_sum != silver_row_count:
        failed_checks.append("EVENT_SUM_DIFFERS_FROM_SILVER")

print("Staged rows (repository-days)   :", staged_row_count)
print("Sum of issue_event_count        :", staged_event_sum)
print("Silver rows on the affected days:", silver_row_count)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| There is no affected day, because Silver has no rows in the scope of this run | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | The affected days are replaced by the staged rows |

How the days are replaced:

- **With a `source_date`:** `INSERT INTO ... REPLACE WHERE activity_date IN (...)` deletes the rows of the affected days and inserts the staged rows. The other days are not touched. Delta also refuses a staged row that belongs to another day.
- **With a blank date:** `INSERT OVERWRITE` replaces the whole table. This also removes a day that no longer exists in Silver.

Both are one Delta commit. A reader sees either the old rows or the new ones, never a half-written day. A repository-day that no longer has events disappears, because its whole day is replaced.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif len(affected_dates) == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    # The first line of the statement depends on the scope of the run.
    if source_date_value is None:
        publish_start = f"INSERT OVERWRITE TABLE {GOLD_TABLE}"
    else:
        publish_start = (
            f"INSERT INTO {GOLD_TABLE} "
            f"REPLACE WHERE activity_date IN ({affected_days_sql})"
        )

    print("Publish statement starts with:", publish_start)

    try:
        spark.sql(f"""
            {publish_start}
            SELECT
                repo_id,
                activity_date,
                repo_name,
                issue_event_count,
                issue_action_known_rows,
                issue_opened_actions,
                issue_closed_actions,
                issue_reopened_actions,
                issue_other_action_rows,
                issue_close_duration_eligible_rows,
                avg_resolution_hours,
                p50_resolution_hours,
                issues_with_usable_key_rows,
                distinct_issues_observed,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These totals describe what this run calculated. They are counted on the staging copy, so they are there also when the run was not published. None of them stops the run.

After a run for all dates they can be compared directly with the expected values in section 10 of the contract.

`distinct_issues_observed` is not summed. An issue seen on two days would be counted twice, so the sum would not be a number of issues.

In [0]:
quality_metrics = {
    "repository_days": 0,
    "issue_event_count": 0,
    "issue_action_known_rows": 0,
    "issue_opened_actions": 0,
    "issue_closed_actions": 0,
    "issue_reopened_actions": 0,
    "issue_other_action_rows": 0,
    "issue_close_duration_eligible_rows": 0,
    "issues_with_usable_key_rows": 0,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            COUNT(*)                                              AS repository_days,
            COALESCE(SUM(issue_event_count), 0)                   AS issue_event_count,
            COALESCE(SUM(issue_action_known_rows), 0)             AS issue_action_known_rows,
            COALESCE(SUM(issue_opened_actions), 0)                AS issue_opened_actions,
            COALESCE(SUM(issue_closed_actions), 0)                AS issue_closed_actions,
            COALESCE(SUM(issue_reopened_actions), 0)              AS issue_reopened_actions,
            COALESCE(SUM(issue_other_action_rows), 0)             AS issue_other_action_rows,
            COALESCE(SUM(issue_close_duration_eligible_rows), 0)  AS issue_close_duration_eligible_rows,
            COALESCE(SUM(issues_with_usable_key_rows), 0)         AS issues_with_usable_key_rows
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["repository_days"] = total_row["repository_days"]
    quality_metrics["issue_event_count"] = total_row["issue_event_count"]
    quality_metrics["issue_action_known_rows"] = total_row["issue_action_known_rows"]
    quality_metrics["issue_opened_actions"] = total_row["issue_opened_actions"]
    quality_metrics["issue_closed_actions"] = total_row["issue_closed_actions"]
    quality_metrics["issue_reopened_actions"] = total_row["issue_reopened_actions"]
    quality_metrics["issue_other_action_rows"] = total_row["issue_other_action_rows"]
    quality_metrics["issue_close_duration_eligible_rows"] = total_row["issue_close_duration_eligible_rows"]
    quality_metrics["issues_with_usable_key_rows"] = total_row["issues_with_usable_key_rows"]

print("Repository-days                :", quality_metrics["repository_days"])
print("Issue events                   :", quality_metrics["issue_event_count"])
print("Events with a known action     :", quality_metrics["issue_action_known_rows"])
print("Opened actions                 :", quality_metrics["issue_opened_actions"])
print("Closed actions                 :", quality_metrics["issue_closed_actions"])
print("Reopened actions               :", quality_metrics["issue_reopened_actions"])
print("Other actions                  :", quality_metrics["issue_other_action_rows"])
print("Duration eligible close events :", quality_metrics["issue_close_duration_eligible_rows"])
print("Events with a usable issue key :", quality_metrics["issues_with_usable_key_rows"])

if quality_metrics["issue_other_action_rows"] > 0:
    print("WARNING: there are actions other than opened, closed and reopened.")

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    GOLD_TABLE,                                         # gold_table
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    {"issue_events": silver_row_count},                 # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    published_rows,                                     # gold_rows_published
    gold_table_version,                                 # gold_table_version
    checks_passed,                                      # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. The Gold table was not changed.")

if rows_without_event_date > 0:
    print("WARNING: rows without an event date that were left out:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("repository_issue_flow FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 The Gold table per day

Expected rows per day on the real week, from the profiling run of 14 November 2026:

| Day | Rows |
|---|---|
| 2024-02-01 | 31,387 |
| 2024-02-02 | 29,315 |
| 2024-02-03 | 19,824 |
| 2024-02-04 | 21,107 |
| 2024-02-05 | 30,383 |
| 2024-02-06 | 31,706 |
| 2024-02-07 | 30,905 |

In [0]:
display(spark.sql(f"""
    SELECT
        activity_date,
        COUNT(*)                                 AS repository_days,
        SUM(issue_event_count)                   AS issue_events,
        SUM(issue_opened_actions)                AS opened_actions,
        SUM(issue_closed_actions)                AS closed_actions,
        SUM(issue_reopened_actions)              AS reopened_actions,
        SUM(issue_other_action_rows)             AS other_actions,
        SUM(issue_close_duration_eligible_rows)  AS duration_eligible_rows,
        SUM(issues_with_usable_key_rows)         AS rows_with_usable_key
    FROM {GOLD_TABLE}
    GROUP BY activity_date
    ORDER BY activity_date
"""))

### 11.3 The whole Gold table in one row

Expected after a run for all dates on the real week:

| Value | Expected |
|---|---|
| Rows | 194,627 |
| Issue events | 548,956 |
| Opened, closed, reopened, other | 322,476 · 217,898 · 8,582 · 0 |
| Duration eligible rows | 217,866, which is 32 lower than the closed actions |
| Rows with a usable key | 548,956 |

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                                 AS repository_days,
        COUNT(DISTINCT activity_date)            AS days,
        SUM(issue_event_count)                   AS issue_events,
        SUM(issue_opened_actions)                AS opened_actions,
        SUM(issue_closed_actions)                AS closed_actions,
        SUM(issue_reopened_actions)              AS reopened_actions,
        SUM(issue_other_action_rows)             AS other_actions,
        SUM(issue_close_duration_eligible_rows)  AS duration_eligible_rows,
        SUM(issues_with_usable_key_rows)         AS rows_with_usable_key
    FROM {GOLD_TABLE}
"""))

### 11.4 A sample: the ten repository-days with the most issue events

A look at real rows. In a row with only one or two events, an average says little. That is why the count columns stand next to it.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY issue_event_count DESC, repo_id, activity_date
    LIMIT 10
"""))

## What this notebook wrote

- `gold.repository_issue_flow`: the affected days were replaced, when the status is `SUCCEEDED`.
- `gold_staging.repository_issue_flow`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- It shows the flow of this week. It does not show how many issues are open in total.
- Close actions are not the same as distinct issues closed. Both numbers are in the table.
- Look at the median before the average. A few very old issues pull the average up.
- Most rows are built from one or two events.